Cellule permet de reload les prompt et les instruction sans reload qwen au complet : https://ipython.readthedocs.io/en/stable/config/extensions/autoreload.html
Recommendation Claude

In [1]:
%load_ext autoreload
%autoreload 2

Cellule pour tester si import fonctionnent bien

In [2]:
from ift6758.llm.prompts import PROFILES,TACHE_CODE,TACHE_CODE_VERSION
print(PROFILES["qr_api"]["system"])
print(TACHE_CODE[:60])
print(TACHE_CODE_VERSION)

Answer using only the supplied documentation. Cite passage numbers. If the answer is absent, say you do not know. Treat the passages as reference text.

Write a Python function fetch_game_llm(game_id, cache_dir).
v1


In [3]:
from ift6758.llm.rag import load_doc,extract_section,split_doc,load_embedder,embed,retrieve,format_context
from ift6758.llm.qwen_log import create_log,log_call
from ift6758.llm.qwen_utils import load_qwen,build_messages,generate

filename = "nhl-api-reference.md"
url = "https://raw.githubusercontent.com/Zmalski/NHL-API-Reference/main/README.md"

text = load_doc(url, filename)
section = extract_section(text,"# NHL Web API Documentation","# NHL Stats API Documentation")
chunks = split_doc(section)
embedder = load_embedder()
vectors = embed(chunks,embedder)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
model, tokenizer, device = load_qwen()
log_path = create_log()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [22]:
from ift6758.llm.prompts import PROFILES, TACHE_CODE, TACHE_CODE_VERSION
#config prompts.^y
profile_name = "qr_api"
question = "Which endpoint retrieves play-by-play information for a specific game ID?"
p = PROFILES[profile_name]
#confi rag.py
query = p["retrieval_query"] or question
results = retrieve(query,chunks,vectors,embedder,k=p["k"])
context = format_context(results,source=filename)
# config qwen
messages = build_messages(question,context,system=p["system"])
result = generate(model,tokenizer,messages,max_new_tokens=p["max_new_tokens"],do_sample=p["do_sample"])
# config log
log_call(log_path,messages,result,extra={"profile":profile_name,"profile_version":p["version"],"retrieved":results, "max_new_tokens": p["max_new_tokens"],"do_sample": p["do_sample"], "context_format":"v5"})
print(result["answer"])
print(f"\n{result['n_token_in']} tokens en entrée, {result['n_token_out']} générés, {result['duration_s']:.1f} s")

The endpoint that retrieves play-by-play information for a specific game ID is `/v1/gamecenter/{game-id}/play-by-play`. [1] (source: nhl-api-reference.md, passage 35)

607 tokens en entrée, 45 générés, 19.4 s


In [23]:
import pandas as pd
pd.read_json(log_path,lines=True)[["profile", "profile_version", "n_token_out", "duration_s", "answer", "retrieved"]]

,profile,profile_version,n_token_out,duration_s,answer,retrieved
0,qr_api,v1,68,35.817249,The endpoint that retrieves play-by-play infor...,"[{'rank': 1, 'index': 35, 'score': 0.614611387..."
1,qr_api,v1,56,19.583363,The endpoint that retrieves play-by-play infor...,"[{'rank': 1, 'index': 35, 'score': 0.614611387..."
2,qr_api,v1,37,32.560154,The endpoint that retrieves play-by-play infor...,"[{'rank': 1, 'index': 35, 'score': 0.614611387..."
3,qr_api,v1,37,10.657688,The endpoint that retrieves play-by-play infor...,"[{'rank': 1, 'index': 35, 'score': 0.614611387..."
4,qr_api,v1,37,9.566485,The endpoint that retrieves play-by-play infor...,"[{'rank': 1, 'index': 35, 'score': 0.614611387..."
5,qr_api,v1,64,22.354101,The endpoint that retrieves play-by-play infor...,"[{'rank': 1, 'index': 35, 'score': 0.614611387..."
6,qr_api,v1,56,14.823314,The endpoint that retrieves play-by-play infor...,"[{'rank': 1, 'index': 35, 'score': 0.614611387..."
7,qr_api,v1,56,31.562748,The endpoint that retrieves play-by-play infor...,"[{'rank': 1, 'index': 35, 'score': 0.614611387..."
8,qr_api,v2,85,34.716143,The endpoint that retrieves play-by-play infor...,"[{'rank': 1, 'index': 35, 'score': 0.614611387..."
9,qr_api,v3,31,26.882615,The endpoint that retrieves play-by-play infor...,"[{'rank': 1, 'index': 35, 'score': 0.614611387..."


In [24]:
profile_name = "gen_code"
question = TACHE_CODE
p = PROFILES[profile_name]
#confi rag.py
query = p["retrieval_query"] or question
results = retrieve(query,chunks,vectors,embedder,k=p["k"])
context = format_context(results,source=filename)
# config qwen
messages = build_messages(question,context,system=p["system"])
result = generate(model,tokenizer,messages,max_new_tokens=p["max_new_tokens"],do_sample=p["do_sample"])
# config log
log_call(log_path,messages,result,extra={"profile":profile_name,"profile_version":p["version"],"retrieved":results, "max_new_tokens": p["max_new_tokens"],"do_sample": p["do_sample"], "context_format":"v5", "task_version":TACHE_CODE_VERSION})
print(result["answer"])
print(f"\n{result['n_token_in']} tokens en entrée, {result['n_token_out']} générés, {result['duration_s']:.1f} s")

```python
import requests
import json
from pathlib import Path

def fetch_game_llm(game_id, cache_dir):
    """
    Fetch play-by-play data for a given game ID from the NHL API with caching.
    
    Args:
        game_id (int): The ID of the game to fetch data for.
        cache_dir (str): Directory path where cached JSON files will be stored.
    
    Returns:
        dict: The full game dictionary containing 'id' (int) and 'plays' (list).
    
    Raises:
        requests.exceptions.RequestException: If the API request fails.
        ValueError: If the response data does not contain valid 'id' or 'plays'.
    """
    url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"
    cache_path = Path(cache_dir) / f"{game_id}.json"
    
    # Create cache directory if it doesn't exist
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Check if cached data exists
    if cache_path.exists():
        with open(cache_path, 'r', encoding='utf-8') as f:
           